In [1]:
from langchain_openai import ChatOpenAI
import os

model = ChatOpenAI(
    model_name="qwen-max",
    openai_api_key = os.getenv("DASHSCOPE_API_KEY"),
    openai_api_base = "https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.9, 
    max_tokens=5000,
    verbose=True
)

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain import hub
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from langchain.embeddings.dashscope import DashScopeEmbeddings
import os
embeddings_model = DashScopeEmbeddings(
    model="text-embedding-v2"
)

In [4]:
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs = {"parse_only": bs4_strainer},
)
docs = loader.load()

In [5]:
print(len(docs[0].page_content))

43130


In [6]:
print(docs[0].page_content[:100])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    add_start_index = True
)
all_splits = text_splitter.split_documents(docs)

In [8]:
print(len(all_splits)) 

66


In [9]:
print(len(all_splits[0].page_content))

969


In [10]:
print(all_splits[0].page_content)

LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:

Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Memory


In [11]:
print(all_splits[0].metadata)

{'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 8}


In [13]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings_model
)

In [14]:
type(vectorstore) 

langchain_chroma.vectorstores.Chroma

In [15]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

In [16]:
retrieved_docs = retriever.invoke("What are the approaches to Task Decomposition?")

In [17]:
print(len(retrieved_docs)) 

6


In [18]:
print(retrieved_docs[0].page_content)

Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.


In [26]:
import os
os.environ["LANGSMITH_API_KEY"] = "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"
os.environ["LANGCHAIN_API_KEY"] = "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"
prompt = hub.pull("rlm/rag-prompt")

/opt/anaconda3/envs/ai-spike/lib/python3.10/site-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [27]:
print(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [28]:
example_messages = prompt.invoke(
    {"context": "filler context", "question": "filler question"}
).to_messages()

In [29]:
print(example_messages[0].content)

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: filler question 
Context: filler context 
Answer:


In [30]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [34]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [35]:
for chunk in rag_chain.stream("What is Task Decomposition?"):
    print(chunk, end="", flush=True)

Task Decomposition is the process of breaking down a complex task into smaller, more manageable steps. This technique helps in making the problem easier to solve and can provide insight into the model's thinking process. It can be achieved through simple prompting, task-specific instructions, or with human input.

In [36]:
for chunk in rag_chain.stream("What is ToT?"):
    print(chunk, end="", flush=True)

ToT, or Tree of Thoughts, is a method that enhances problem-solving by breaking down complex tasks into multiple steps and exploring various reasoning paths at each step, creating a tree-like structure. This approach allows for the generation of multiple thoughts per step, which can be navigated using search strategies like BFS or DFS. The process helps in evaluating different solutions through a classifier or majority vote, making it effective for tackling complicated tasks.